# 4 — Broad lineage: ordered residual gating (5 compartments + Other)

Types every cell by the **first matching gate in priority order**, each gate running on the RESIDUAL of the
prior: **Epithelial** (`E_cadherin`; hormone⁺ sub-branch = **Endocrine** β/α/δ) → **Endothelial** (`CD31`)
→ **Neural** (`B3TUBB`) → **Immune** (marker union) → **Mesenchymal** (`SMA` then `Vimentin`) → **Other**
(failed every gate — real panel gaps, NOT force-assigned). `_pos` comes straight from RESTORE (no `_norm`
floor). **Begin with the composition trace**, then run the assignment.

In [ ]:
# Parameters for this step (self-contained -- no config.ini). Edit the paths for your machine.
%load_ext autoreload
%autoreload 2                                         # pick up edits to the phenocycler package without a kernel restart
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `phenocycler` importable from notebooks/
from phenocycler import PipelineConfig

REPO = pathlib.Path.cwd().resolve().parents[1]        # Islet-Explorer-Senior (parent of the submodule; data/ lives here)

# Run on a subset of donors (None = every donor under data/cells/donor_id=*).
DONORS = None            # e.g. ["6374", "6380"] to iterate on a few

# Composition-trace diagnostic knobs (recomputes gate thresholds like RESTORE, on the trace donor).
THRESHOLD_STAT = "mean3sd"
MODEL          = "SSC"
IDX_FLOOR_Q    = 0.5
SUBSAMPLE      = 8000
SEED           = 0

cfg = PipelineConfig(
    data_dir       = REPO / "data",
    donor_metadata = pathlib.Path("/home/smith6jt/IO60panc2nd/donor_metadata_panc.xlsx"),   # disease-status for the composition validation
    n_jobs = 8,                 # per-donor pool for the vectorized ordered-residual gating
)
donors = DONORS or cfg.discover_donors()
print(f"donors: {len(donors)} " + ("(subset)" if DONORS else "(all)") + f" -> {donors[:6]}" + (" ..." if len(donors) > 6 else ""))

## Diagnostics — threshold → composition trace (begin here)

The payoff of the RESTORE thresholds: the `_pos` calls feed the ordered-residual gating tree. Here we
threshold an *illustrative subset* of gate markers on the trace donor with the notebook's current settings
(so changing the floor/statistic flows all the way to composition), assign compartments in priority order,
and split hormone⁺ Epithelial into Endocrine. ~12 SSC fits/donor ≈ a few minutes; the real pipeline (below)
runs every gate marker + the robust guard cohort-wide.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from phenocycler.diagnostics import read_donor, idx_select, neg_stat, fit_clusters, status_map
from phenocycler.config import COMPARTMENT_ORDER, OTHER_LABEL, MARKER_PAIRS
%matplotlib inline               # AFTER the imports: phenocycler.restore sets a non-interactive backend on import

DIAG_DONOR = donors[0]
REFMAP = {t: r for t, r in MARKER_PAIRS}                        # target -> its mutually-exclusive reference
GATE_MK = {"Epithelial": ["E_cadherin"], "Endothelial": ["CD31"], "Neural": ["B3TUBB"],
           "Immune": ["CD3e", "CD20", "CD68", "CD163"], "Mesenchymal": ["SMA", "Vimentin"]}   # illustrative subset
ENDO = ["INS", "GCG", "SST"]
need = sorted(set(sum(GATE_MK.values(), []) + ENDO))
dl = read_donor(cfg, DIAG_DONOR, sorted(set(need) | {REFMAP[m] for m in need}))
posd = {}
for m in need:                                                 # threshold each gate marker (the notebook's method)
    Tm = dl[m].to_numpy(float); Rm = dl[REFMAP[m]].to_numpy(float)
    sel, _, _ = idx_select(Tm, Rm, IDX_FLOOR_Q)
    cl, lb, ng = fit_clusters(np.column_stack([Tm[sel], Rm[sel]]), MODEL, subsample=SUBSAMPLE, seed=SEED)
    posd[m] = Tm >= neg_stat(cl[lb == ng][:, 0], THRESHOLD_STAT)
comp = np.full(len(dl), OTHER_LABEL, dtype=object)             # ordered-residual gating (priority order)
for c in COMPARTMENT_ORDER:
    gate = np.zeros(len(dl), bool)
    for m in GATE_MK.get(c, []): gate = gate | posd.get(m, np.zeros(len(dl), bool))
    comp[(comp == OTHER_LABEL) & gate] = c
horm = np.zeros(len(dl), bool)
for m in ENDO: horm = horm | posd.get(m, np.zeros(len(dl), bool))
comp[(comp == "Epithelial") & horm] = "Endocrine"             # hormone+ sub-branch of Epithelial
vc = pd.Series(comp).value_counts(normalize=True).mul(100)
vc = vc.reindex([c for c in ["Epithelial", "Endocrine", "Endothelial", "Neural", "Immune", "Mesenchymal", OTHER_LABEL]
                 if c in vc.index])
fig, a = plt.subplots(figsize=(8.5, 4)); a.bar(vc.index, vc.values, color="C0")
for x, v in enumerate(vc.values): a.text(x, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
a.set_ylabel("% of cells"); a.tick_params(axis="x", rotation=20)
a.set_title(f"{DIAG_DONOR} ({status_map(cfg).get(DIAG_DONOR,'?')}) — broad composition [illustrative gate subset]")
fig.tight_layout(); plt.show()
print("composition (%):", {k: round(v, 1) for k, v in vc.items()})
print("(subset gates -> 'Other' inflated vs the full pipeline; change DONORS to an ND vs T1D to see "
      "Endocrine fall / Immune rise)")

## Run — broad lineage on the (subset of) donors

In [ ]:
from phenocycler.lineage import run_lineage
C = run_lineage(cfg, donors=DONORS, n_jobs=cfg.n_jobs)   # donors=None -> all
C

## Post-run validation

Cells typed, the `Other` fraction (real panel gaps, not force-assigned), per-compartment counts, and
composition (% of cells) by disease status — read directly from the `compartment` column. Endocrine should
fall and Immune rise ND → Aab+ → T1D.

In [ ]:
from collections import Counter
import pandas as pd
import pyarrow.dataset as ds
from phenocycler.lineage import status_map
from phenocycler.config import COMPARTMENT_ORDER, OTHER_LABEL, STATUS_ORDER

_ORDER = COMPARTMENT_ORDER + [OTHER_LABEL]
smap = status_map(cfg)
total, other = 0, 0
comp_counts, status_comp = Counter(), {}
for d in donors:
    t = ds.dataset(cfg.broad_dir / f"donor_id={d}", format="parquet").to_table(columns=["compartment"]).to_pandas()
    total += len(t); other += int((t["compartment"] == OTHER_LABEL).sum())
    vc = t["compartment"].value_counts().to_dict()
    comp_counts.update(vc)
    status_comp.setdefault(smap.get(d, "?"), Counter()).update(vc)
print(f"Total cells typed: {total:,}")
print(f"{OTHER_LABEL} (failed every gate, real panel gaps): {100*other/total:.1f}%\n")
vcs = pd.Series(comp_counts).reindex(_ORDER).fillna(0).astype(int)
display(vcs.to_frame("cells").assign(**{"%": (100 * vcs / total).round(2)}))
if smap:
    comp = pd.DataFrame(status_comp).T.fillna(0)
    comp = comp.div(comp.sum(axis=1), axis=0).mul(100).round(1)
    order = [s for s in STATUS_ORDER if s in comp.index] + [s for s in comp.index if s not in STATUS_ORDER]
    comp = comp.reindex(index=order, columns=[c for c in _ORDER if c in comp.columns])
    print("Composition (% of cells) by disease status:")
    display(comp)

In [ ]:
from IPython.display import Image, display
fig = cfg.phenotype_dir / "broad_lineage_composition.png"
display(Image(filename=str(fig))) if fig.exists() else print(f"({fig.name} not found — run the lineage step)")